In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
# 改了 src/ 里的 .py，notebook 不用重启就能用上新代码
from signal_mysql import database as mysql
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, db, geo, privacy, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT, AREA_COLOR, SEG_COLOR  # 配色常量
viz.apply_theme()                           # 字体、配色、坐标轴样式：一行
con = mysql.connect(config.DB_SCHEMA)       # 路径从包里来：notebook 在哪个文件夹打开都找得到库
DAY = config.ANALYSIS_DAY                   # 分析日期（口径常量）
NB = "D13_测试应用层与报告"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"    # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path
from signal_insight import __main__ as cli_main, ads, dims, dws, ingest, pipeline
def cli(*args):
    """和在终端里敲 python -m signal_insight … 一样。"""
    return cli_main.main(list(args))
settings = config.load_settings()
wh = db.connect(config.WAREHOUSE_SCHEMA)
reg = ingest.registry(wh).set_index("dt")
assert reg.loc[DAY, "pipeline_status"] == "done", "先完成 D11、D12"
reg[["is_synthetic", "gate_status", "pipeline_status"]]

In [ ]:
import ast
counts = {}
for folder in ["unit", "contract", "integration"]:
    files = sorted((config.PROJECT_ROOT / "tests" / folder).glob("test_*.py"))
    n = sum(isinstance(x, ast.FunctionDef) and x.name.startswith("test_")
            for f in files for x in ast.walk(ast.parse(f.read_text(encoding="utf-8"))))
    counts[folder] = {"文件数": len(files), "测试函数数": n}
print(pd.DataFrame(counts).T)
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/unit", "tests/integration"],
                   cwd=config.PROJECT_ROOT, capture_output=True, text=True, encoding="utf-8")
print("没装 pytest：python -m pip install -e \".[dev]\"" if "No module named pytest" in r.stderr
      else (r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:]))

In [ ]:
# ════ 从 bug 到测试：修 bug 的标准动作 ════
def classify_v1(dist_km):
    d = round(dist_km, 2)                          # D08 的写法：先取整
    return "周边居民" if d < 1.5 else "市内来访"


def classify_v2(dist_km):
    return "周边居民" if dist_km < 1.5 else "市内来访"   # 先判断，展示时再取整


def test_resident_boundary(classify):
    assert classify(1.499) == "周边居民", "1.499 公里应算周边居民"
    assert classify(1.5) == "市内来访"
    assert classify(0.2) == "周边居民"


for f in (classify_v1, classify_v2):
    try:
        test_resident_boundary(f); print(f"{f.__name__}：通过")
    except AssertionError as e:
        print(f"{f.__name__}：失败 —— {e}")

In [ ]:
# 由配置渲染的口径表，和手写的逐字对比（D13 起追加了区域级口径，只比这里手写的 9 条）
dim_metric = pd.DataFrame([
    ("西塔停留", "在西塔停下来待了一段时间", "停留质心距西塔中心 ≤ 半径，且持续 ≥ 最短时长", "半径 500 米；最短 30 分钟", "dws_stay"),
    ("傍晚访客", "国庆当天傍晚后在西塔待过的人", "有西塔停留，且该停留结束时刻 ≥ 17:00", "17:00", "dws_stay"),
    ("西塔停留时长", "在西塔待了多久", "当天所有西塔停留时长之和；每段按首条到末条信令计", "23 点后仍在者为删失，只是下限", "dws_user_day"),
    ("过夜地", "凌晨待得最久的地方（不等于家）", "与 0:00–6:00 重叠最长且重叠 ≥ 60 分钟的停留", "60 分钟", "dws_night"),
    ("客源距离", "从多远来", "过夜地质心到西塔中心的直线距离", "直线距离，不是路程", "dws_night"),
    ("客群标签", "是哪类人", "按过夜地：不明 / 市外 / 1.5 公里内为周边居民 / 其余为市内来访", "1.5 公里", "dws_user_day"),
    ("客群标签（修正）", "是哪类人，含当天从外地进城的人",
     "当天第一条信令在市域外、之后进入市域，或过夜地在市外：外地来客；其余同客群标签", "—", "dws_user_day.segment_v2"),
    ("出行", "从一个地方到另一个地方", "相邻两个停留质心相距 ≥ 500 米", "500 米", "dws_trip"),
    ("商圈联动率", "西塔访客还去了哪个商圈", "傍晚访客中，当天在该商圈有停留的人的占比", "商圈边界用 v2", "dws_user_day"),
], columns=["metric", "meaning", "definition", "params", "source_table"])
generated = dims.build_dim_metric(settings)
pd.testing.assert_frame_equal(generated.head(len(dim_metric)).drop(columns="version"), dim_metric.drop(columns="version"))
db.replace_table(con, generated, "dim_metric")
trial = {**settings, "hotspot": {**settings["hotspot"], "radius_m": 1000}}
print(dims.build_dim_metric(trial).loc[0, "params"])
generated

In [ ]:
# 对账：规则收进函数以后，D08 宽表的客群一行不变；边界就是会写进测试的那两行
ud_all = mysql.read_frame("SELECT * FROM dws_user_day", con)
assert (dws.classify_segment(ud_all, settings) == ud_all["segment"]).all()
assert list(dws.segment_by_night([1.499, 1.5], ["和平区", "和平区"], 1.5)) == ["周边居民", "市内来访"]
print(f"✅ 对账通过：D08 宽表 {len(ud_all):,} 人的客群不变；1.499 公里 = 周边居民，1.5 公里 = 市内来访")

In [ ]:
# 维表多了 dim_area、dim_area_polygon；按日期跑完全部分区，长出应用层
cli("dims")
cli("run", "--start", "2018-10-01", "--end", "2018-10-06")

In [ ]:
from signal_mysql import database as mysql
USE = {"ads_area_hourly": "节律页、总览、报告图 A01", "ads_area_daily": "总览 KPI、趋势、报告骨架",
       "ads_area_segment": "客源页、报告图 A02", "ads_area_radius": "客源页、报告图 A03",
       "ads_area_origin": "客源页、报告图 A04", "ads_area_linkage": "联动页、报告图 A05",
       "ads_hex_hourly": "空间页、报告图 A06", "ads_od_district": "区县流向（D14 选讲）",
       "ads_dq_partition": "质量页、质量报告", "ads_dq_rule": "质量页、质量报告",
       "ads_dq_clean_steps": "质量页、清洗说明"}
rows = []
for t, use in USE.items():
    cols = [r[1] for r in mysql.column_result(wh, t)]
    n = wh.execute(f"SELECT COUNT(*) FROM {t} WHERE dt = ?", (DAY,)).fetchone()[0]
    rows.append({"表": t, "10-03 行数": n, "列数": len(cols), "带模拟标记": "is_synthetic" in cols,
                 "带抑制标记": "suppressed" in cols, "使用方": use})
pd.DataFrame(rows)

In [ ]:
from signal_mysql import database as mysql
# 区域可以重叠：17 点同时落在西塔 500 米圈和太原街商圈里的“人 × 小时”
dim_area, area_polys = pipeline.read_areas(wh)
uh17 = mysql.read_frame("SELECT imsi, hour, lng, lat FROM dws_user_hour WHERE dt = ? AND hour = 17", wh, params=(DAY,))
m = geo.area_membership(uh17["lng"], uh17["lat"], dim_area, area_polys)
both = m.groupby("idx")["area"].apply(set)
print("17 点同时属于西塔和太原街商圈的“人 × 小时”：", int(both.apply(lambda s: {"西塔", "太原街商圈"} <= s).sum()))

In [ ]:
from signal_mysql import database as mysql
# 对账：区域表里的西塔 = D08 的西塔；互补抑制之后，没有哪一组只藏了一个格子
daily = mysql.read_frame("SELECT * FROM ads_area_daily WHERE dt = ?", wh, params=(DAY,)).set_index("area")
ev_d08 = con.execute("SELECT COUNT(*) FROM dws_user_day WHERE is_evening_visitor").fetchone()[0]
g = mysql.read_frame("SELECT dt, area, SUM(suppressed) s, COUNT(*) n FROM ads_area_segment GROUP BY dt, area", wh)
assert daily.loc["西塔", "evening_visitors"] == ev_d08 == 3_310
assert not ((g["s"] == 1) & (g["n"] > 1)).any()
print(f"✅ 对账通过：ads_area_daily 西塔傍晚访客 {daily.loc['西塔', 'evening_visitors']:,.0f} = D08 的 {ev_d08:,}；没有“只藏一个格子”的组")

In [ ]:
from signal_mysql import database as mysql
# 客户追加需求：“青年大街地铁站周边 500 米也看一下。”只在内存里加一项配置，计算模块一行不改。
st_ = mysql.read_frame("SELECT name, lng, lat FROM ods_poi_transport WHERE name LIKE '青年大街%' AND type = '地铁'", con).iloc[0]
new_area = {"name": "青年大街站", "kind": "circle", "lng": float(st_["lng"]), "lat": float(st_["lat"]), "radius_m": 500,
            "note": "地铁 1、2 号线换乘站，坐标取自 ods_poi_transport"}
settings_new = {**settings, "areas": settings["areas"] + [new_area]}
aois = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_aoi_v2", con), "name")
dim_new, poly_new = dims.build_dim_area(settings_new, aois)
dim_new.tail(3)

In [ ]:
# 用新的区域维表，把 10-03 的停留重新做一次“区域归属”和“用户 × 区域”汇总——计算模块一行没改
stay = db.read_partition(wh, "dws_stay", DAY, parse_dates=["start_time", "end_time"])
user_day = db.read_partition(wh, "dws_user_day", DAY)
night = db.read_partition(wh, "dws_night", DAY)
sa = dws.build_stay_area(stay, dim_new, geo.rings_from_points(poly_new, "area"))
uad = dws.build_user_area_day(stay, sa, user_day, night, dim_new, DAY, settings_new)
daily_new = ads.build_area_daily(uad, dim_new, settings_new).set_index("area")
qn = daily_new.loc["青年大街站"]
print(f"青年大街站：当天访客 {qn['visitors_all']:.0f} 人，傍晚访客 {qn['evening_visitors']:.0f} 人")
assert (qn["visitors_all"], qn["evening_visitors"]) == (37, 18)      # 对账：只改配置，37 人、傍晚 18 人
daily_new.loc[["西塔", "青年大街站"], ["visitors_all", "evening_visitors", "stay_min_p50", "night_dist_km_p50", "inbound_share"]]

In [ ]:
from signal_mysql import database as mysql
# ════ 差分攻击：为什么“抑制小格子”还不够 ════
k = settings["privacy"]["min_cell_users"]
bands = settings["ads"]["radius_bands_km"]
labels = ads.radius_band_labels(bands)
ev = uad[uad["evening_visitor"] & (uad["area"] == "中街商圈")].copy()
ev["band"] = pd.cut(ev["night_dist_km"], [-np.inf, *bands, np.inf], labels=labels[:-1], right=False).astype(object)
ev["band"] = ev["band"].where(ev["night_dist_km"].notna(), labels[-1])
raw = ev.groupby("band").size().reindex(labels).rename("真实人数")
naive = raw.where(raw >= k)                                      # 只做一级抑制
total = int(daily.loc["中街商圈", "evening_visitors"])             # 总览页公布的总数
print(f"中街商圈傍晚访客（公布）{total} 人；一级抑制藏了 {int(naive.isna().sum())} 档")
print(f"攻击：{total} − 其他各档之和 {int(naive.sum())} = {total - int(naive.sum())}  ← 就是被藏的“{naive[naive.isna()].index[0]}”的真实人数")
safe = mysql.read_frame("SELECT band, users, cum_share FROM ads_area_radius WHERE dt = ? AND area = '中街商圈' ORDER BY band_order",
                        wh, params=(DAY,)).set_index("band")
print(f"流水线（互补抑制）：藏了 {int(safe['users'].isna().sum())} 档，差出来的 {total - int(safe['users'].sum())} 是两档之和，拆不开")
assert (total, total - int(naive.sum()), total - int(safe["users"].sum())) == (315, 3, 20)
pd.concat([raw, naive.rename("只做一级抑制"), safe["users"].rename("流水线（互补抑制）"),
           safe["cum_share"].rename("累计占比（流水线）")], axis=1)

In [ ]:
from signal_mysql import database as mysql
# ── 出图 D13_02：抑制的代价 ───────────────────────────────────────────
sup = {}
for t in ["ads_area_hourly", "ads_area_segment", "ads_area_radius", "ads_area_origin"]:
    x = mysql.read_frame(f"SELECT suppressed FROM {t} WHERE dt = ?", wh, params=(DAY,))
    sup[t] = (int(x["suppressed"].sum()), len(x))
l_ = mysql.read_frame("SELECT users_any FROM ads_area_linkage WHERE dt = ?", wh, params=(DAY,))
sup["ads_area_linkage"] = (int(l_["users_any"].isna().sum()), len(l_))   # 一对区域算一格
s_ = pd.Series({k_: v[0] / v[1] for k_, v in sup.items()}).sort_values()
fig, ax = plt.subplots(figsize=(10, 5.6))
bars = ax.barh(s_.index, s_.values, color=[MAIN if k_ == s_.idxmax() else GRAY for k_ in s_.index], height=0.6)
ax.bar_label(bars, labels=[f"{v:.0%}（{sup[k_][0]}/{sup[k_][1]} 格）" for k_, v in s_.items()], padding=4, fontsize=10)
ax.set_xlim(0, 1); ax.set_xticks([]); ax.grid(False); ax.tick_params(axis="y", labelsize=10)
viz.finish(fig, f"{s_.idxmax().replace('ads_area_', '')} 表 {s_.max():.0%} 的格子被抑制：区域越小、切得越细，能展示的越少",
           "10-03 各 ADS 表中少于 5 人（含互补抑制）而置空的格子比例；世博园、棋盘山这类人少的区域大部分格子为空；\n"
           "这是隐私保护的代价：要展示小区域，就要合并类别或放宽时间粒度",
           SOURCE, "D13_02_抑制比例.png")

In [ ]:
from signal_mysql import database as mysql
import importlib.util
import time
# 迷你世界是测试代码，不是包的一部分：按文件路径加载，不改 sys.path
spec = importlib.util.spec_from_file_location("miniworld", config.PROJECT_ROOT / "tests" / "integration" / "miniworld.py")
mw = importlib.util.module_from_spec(spec); spec.loader.exec_module(mw)
sys.modules["miniworld"] = mw                         # 测试文件里的 from miniworld import … 也找得到它
mwh, msettings = mw.build_warehouse(Path(tempfile.mkdtemp()))
t0 = time.perf_counter()
mcon = mysql.connect(mwh)
for d in [mw.D1, mw.D2]:
    pipeline.run_partition(mcon, msettings, d)
mw_secs = time.perf_counter() - t0
summary = mysql.read_frame("SELECT dt, COUNT(*) AS 停留, SUM(cross_day) AS 跨午夜, ROUND(AVG(duration_min)) AS 平均分钟 FROM dws_stay GROUP BY dt", mcon)
mcon.close()                                          # 测试里一定要关连接：Windows 不允许删还开着的文件
print(f"迷你世界两天全流程：{mw_secs:.1f} 秒")
summary

In [ ]:
# ── 出图 D13_01：迷你世界长什么样 ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5.6))
sq = np.array([(123.3, 41.7), (123.6, 41.7), (123.6, 41.9), (123.3, 41.9), (123.3, 41.7)])
ax.plot(sq[:, 0], sq[:, 1], color=FAINT, lw=1)
ax.text(0.01, 0.02, "灰色框线 = 测试区（区县）边界", transform=ax.transAxes, color=MUTED, fontsize=9.5)
mall = np.array([(123.445, 41.795), (123.455, 41.795), (123.455, 41.805), (123.445, 41.805), (123.445, 41.795)])
ax.fill(mall[:, 0], mall[:, 1], color=viz.area_color("中街商圈"), alpha=0.25)
ax.text(123.45, 41.807, "测试商圈", ha="center", color=viz.area_color("中街商圈"), fontsize=10)
ang = np.linspace(0, 2 * np.pi, 100)
ax.plot(123.4 + 500 * np.cos(ang) / (111320 * np.cos(np.radians(41.8))), 41.8 + 500 * np.sin(ang) / 110574, ls="--", color=MAIN)
ax.text(123.4, 41.806, "测试热点 500 米", ha="center", color=MAIN, fontsize=10)
recs = pd.DataFrame(mw.user_day_records("u", mw.D1), columns=["ts", "imsi", "lac", "ci", *list("abcdef")])
recs["key"] = recs["lac"].astype(int).astype(str) + "-" + recs["ci"].astype(int).astype(str)
offsets = {"1-1": (-10, 14, "right"), "1-2": (10, -18, "left"), "1-3": (10, -18, "left")}
for key, (lng, lat) in mw.CELLS.items():
    if lng < 123:
        continue                                     # 市外基站不在图里
    n = int((recs["key"] == key).sum())
    ax.scatter(lng, lat, s=40 + n * 3, color=INK, zorder=3)
    dx, dy, ha = offsets[key]
    ax.annotate(f"基站 {key}（每人 {n} 条/天）", (lng, lat), xytext=(dx, dy), textcoords="offset points", fontsize=9.5, ha=ha)
ax.annotate("", (123.4495, 41.8003), (123.4035, 41.8003),
            arrowprops=dict(arrowstyle="->", color=MUTED, lw=1, connectionstyle="arc3,rad=-0.25"))
ax.text(123.426, 41.8098, "10 点去商圈", ha="center", fontsize=9, color=MUTED)
ax.annotate("", (123.4015, 41.7997), (123.4485, 41.7997),
            arrowprops=dict(arrowstyle="->", color=MUTED, lw=1, connectionstyle="arc3,rad=-0.25"))
ax.text(123.426, 41.7932, "18 点回住处", ha="center", fontsize=9, color=MUTED)
ax.set_xlim(123.395, 123.46); ax.set_ylim(41.785, 41.813)
ax.set_xlabel("经度"); ax.set_ylabel("纬度")
viz.finish(fig, "迷你世界：4 个基站、1 个商圈、20 个人，每个数都能手算",
           "0–9 点在 A、B 之间乒乓（人没动），10–16 点在 C（商圈里），18 点以后回 A；灰色框线是区县，蓝色虚线是热点 500 米",
           SOURCE, "D13_01_迷你世界.png")